In [ ]:
!pip install tensorflow openpyxl joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving ML_Ready_Air_Quality_Dataset.csv.xlsx to ML_Ready_Air_Quality_Dataset.csv (1).xlsx


In [ ]:
filename = list(uploaded.keys())[0]

df = pd.read_excel("ML_Ready_Air_Quality_Dataset.csv (1).xlsx")

print("Dataset Loaded Successfully!")

print("Shape :", df.shape)

Dataset Loaded Successfully!
Shape : (1048575, 25)


In [ ]:
df.head()

,Timestamp,PM2.5,PM10,Nitric Oxide,Nitrogen Dioxide,Nitrogen Oxides,Ammonia,Sulfur Dioxide,Carbon Monoxide,Ozone,...,City,Latitude,Longitude,Calculated_AQI,Month,Hour,DayOfWeek,Is_Weekend,State_Encoded,City_Encoded
0,2017-01-01 00:00:00,162.00,227.00,35.43,84.29,123.52,101.19,21.52,1.95,8.33,...,"Anand Vihar ( , )",28.6508,77.3152,90.105556,1,0,6,1,6,4
1,2017-01-01 01:00:00,165.17,226.17,49.52,83.79,142.28,124.92,21.76,2.16,8.98,...,"Anand Vihar ( , )",28.6508,77.3152,90.105556,1,1,6,1,6,4
2,2017-01-01 02:00:00,176.67,215.33,48.78,76.69,134.95,119.38,21.82,1.96,10.48,...,"Anand Vihar ( , )",28.6508,77.3152,90.105556,1,2,6,1,6,4
3,2017-01-01 03:00:00,150.67,177.50,26.71,77.01,105.08,111.60,21.70,1.74,12.88,...,"Anand Vihar ( , )",28.6508,77.3152,90.105556,1,3,6,1,6,4
4,2017-01-01 04:00:00,129.50,174.50,115.47,63.65,214.40,73.84,22.54,2.30,14.71,...,"Anand Vihar ( , )",28.6508,77.3152,90.105556,1,4,6,1,6,4


In [ ]:
print(df.isnull().sum())

Timestamp              0
PM2.5                  0
PM10                   0
Nitric Oxide           0
Nitrogen Dioxide       0
Nitrogen Oxides        0
Ammonia                0
Sulfur Dioxide         0
Carbon Monoxide        0
Ozone                  0
Ambient Temperature    0
Relative Humidity      0
Solar Radiation        0
Rainfall               0
State                  0
City                   0
Latitude               0
Longitude              0
Calculated_AQI         0
Month                  0
Hour                   0
DayOfWeek              0
Is_Weekend             0
State_Encoded          0
City_Encoded           0
dtype: int64


In [ ]:
df = df.dropna()

print("Remaining rows :", len(df))

Remaining rows : 1048575


In [ ]:
print("Duplicate Rows :", df.duplicated().sum())

Duplicate Rows : 0


In [ ]:
df = df.drop_duplicates()

print("Dataset Shape :", df.shape)

Dataset Shape : (1048575, 25)


In [ ]:
features = [

    'PM2.5',

    'PM10',

    'Nitric Oxide',

    'Nitrogen Dioxide',

    'Nitrogen Oxides',

    'Ammonia',

    'Sulfur Dioxide',

    'Carbon Monoxide',

    'Ozone',

    'Ambient Temperature',

    'Relative Humidity',

    'Solar Radiation',

    'Rainfall',

    'Latitude',

    'Longitude',

    'State_Encoded',

    'City_Encoded',

    'Month',

    'Hour',

    'DayOfWeek',

    'Is_Weekend'

]

target = "Calculated_AQI"

In [ ]:
X = df[features]

y = df[target]

print(X.shape)

print(y.shape)

(1048575, 21)
(1048575,)


In [ ]:
df[['City', 'Timestamp']].head(10)

,City,Timestamp
0,"Anand Vihar ( , )",2017-01-01 00:00:00
1,"Anand Vihar ( , )",2017-01-01 01:00:00
2,"Anand Vihar ( , )",2017-01-01 02:00:00
3,"Anand Vihar ( , )",2017-01-01 03:00:00
4,"Anand Vihar ( , )",2017-01-01 04:00:00
5,"Anand Vihar ( , )",2017-01-01 05:00:00
6,"Anand Vihar ( , )",2017-01-01 06:00:00
7,"Anand Vihar ( , )",2017-01-01 07:00:00
8,"Anand Vihar ( , )",2017-01-01 08:00:00
9,"Anand Vihar ( , )",2017-01-01 09:00:00


In [ ]:
df = df.sort_values(by=["City", "Timestamp"]).reset_index(drop=True)

print("Dataset sorted successfully!")

Dataset sorted successfully!


In [ ]:
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

X_scaled = feature_scaler.fit_transform(X)

y_scaled = target_scaler.fit_transform(
    y.values.reshape(-1,1)
)

print("Scaling Completed!")

Scaling Completed!


In [ ]:
joblib.dump(feature_scaler, "feature_scaler.pkl")
joblib.dump(target_scaler, "target_scaler.pkl")

print("Scalers Saved Successfully!")

Scalers Saved Successfully!


In [ ]:
TIME_STEPS = 24

In [ ]:
# ============================================
# Create Train/Test Sequences City-wise
# ============================================

X_train = []
y_train = []

X_test = []
y_test = []

TIME_STEPS = 24

cities = sorted(df["City"].unique())

print("Total Cities :", len(cities))

for city in cities:

    city_df = df[df["City"] == city].copy()

    city_df = city_df.sort_values("Timestamp").reset_index(drop=True)

    # Scale this city's data
    X_city = feature_scaler.transform(city_df[features])

    y_city = target_scaler.transform(
        city_df[target].values.reshape(-1,1)
    )

    # Create sequences
    X_seq = []
    y_seq = []

    for i in range(TIME_STEPS, len(city_df)):
        X_seq.append(X_city[i-TIME_STEPS:i])
        y_seq.append(y_city[i])

    X_seq = np.array(X_seq)
    y_seq = np.array(y_seq)

    # 80-20 split for this city
    split = int(len(X_seq) * 0.8)

    X_train.extend(X_seq[:split])
    y_train.extend(y_seq[:split])

    X_test.extend(X_seq[split:])
    y_test.extend(y_seq[split:])

print("Sequence Creation Completed!")

Total Cities : 23
Sequence Creation Completed!


In [ ]:
X_train = np.array(X_train)
X_test = np.array(X_test)

y_train = np.array(y_train)
y_test = np.array(y_test)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

Training Shape : (838411, 24, 21)
Testing Shape  : (209612, 24, 21)


In [ ]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (838411, 24, 21)
X_test  : (209612, 24, 21)
y_train : (838411, 1)
y_test  : (209612, 1)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional

model = Sequential()

model.add(
    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        ),
        input_shape=(TIME_STEPS, X_train.shape[2])
    )
)

model.add(Dropout(0.3))

model.add(
    Bidirectional(
        LSTM(32)
    )
)

model.add(Dropout(0.3))

model.add(Dense(32, activation="relu"))

model.add(Dense(16, activation="relu"))

model.add(Dense(1))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 24, 128)        │        44,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 87,873 (343.25 KB)

 Trainable params: 87,873 (343.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

print("Model Compiled Successfully!")

Model Compiled Successfully!


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=7,

    restore_best_weights=True,

    verbose=1

)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(

    "Best_AQI_BiLSTM.keras",

    monitor="val_loss",

    save_best_only=True,

    verbose=1

)

In [ ]:
history = model.fit(

    X_train,

    y_train,

    validation_data=(X_test, y_test),

    epochs=30,

    batch_size=64,

    callbacks=[early_stop, checkpoint],

    verbose=1

)

Epoch 1/30
13101/13101 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0031 - mae: 0.0321
Epoch 1: val_loss improved from None to 0.00296, saving model to Best_AQI_BiLSTM.keras

Epoch 1: finished saving model to Best_AQI_BiLSTM.keras
13101/13101 ━━━━━━━━━━━━━━━━━━━━ 185s 14ms/step - loss: 0.0014 - mae: 0.0216 - val_loss: 0.0030 - val_mae: 0.0370
Epoch 2/30
13100/13101 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.3582e-04 - mae: 0.0139
Epoch 2: val_loss improved from 0.00296 to 0.00253, saving model to Best_AQI_BiLSTM.keras

Epoch 2: finished saving model to Best_AQI_BiLSTM.keras
13101/13101 ━━━━━━━━━━━━━━━━━━━━ 178s 14ms/step - loss: 6.0108e-04 - mae: 0.0132 - val_loss: 0.0025 - val_mae: 0.0297
Epoch 3/30
13099/13101 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 5.2541e-04 - mae: 0.0119
Epoch 3: val_loss did not improve from 0.00253
13101/13101 ━━━━━━━━━━━━━━━━━━━━ 177s 13ms/step - loss: 5.0500e-04 - mae: 0.0115 - val_loss: 0.0031 - val_mae: 0.0374
Epoch 4/30
13098/13101 ━━━━━━━━━━━━━━━━━━━━ 0s

In [ ]:
y_pred = model.predict(X_test)

6551/6551 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step


In [ ]:
y_test_actual = target_scaler.inverse_transform(y_test)

y_pred_actual = target_scaler.inverse_transform(y_pred)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import numpy as np

mae = mean_absolute_error(y_test_actual, y_pred_actual)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))

r2 = r2_score(y_test_actual, y_pred_actual)

print("="*45)

print("FINAL MODEL PERFORMANCE")

print("="*45)

print(f"MAE  : {mae:.2f}")

print(f"RMSE : {rmse:.2f}")

print(f"R²   : {r2:.4f}")

FINAL MODEL PERFORMANCE
MAE  : 12.02
RMSE : 20.39
R²   : 0.9627


In [ ]:
df["Calculated_AQI"].describe()

,Calculated_AQI
count,1.048575e+06
mean,1.386243e+02
std,1.097796e+02
min,1.219540e-01
25%,5.665292e+01
50%,1.000671e+02
75%,1.870770e+02
max,4.055918e+02


In [ ]:
from sklearn.preprocessing import LabelEncoder

city_encoder = LabelEncoder()
state_encoder = LabelEncoder()

city_encoder.fit(df["City"])
state_encoder.fit(df["State"])

LabelEncoder()

In [ ]:
import joblib

# Save model
model.save("AQI_BiLSTM_Model.keras")

# Save scalers
joblib.dump(feature_scaler, "feature_scaler.pkl")
joblib.dump(target_scaler, "target_scaler.pkl")

# Save encoders
joblib.dump(city_encoder, "city_encoder.pkl")
joblib.dump(state_encoder, "state_encoder.pkl")

print("Everything Saved Successfully")

Everything Saved Successfully


In [ ]:
from google.colab import files

files.download("AQI_BiLSTM_Model.keras")
files.download("feature_scaler.pkl")
files.download("target_scaler.pkl")
files.download("city_encoder.pkl")
files.download("state_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>